# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook guides you through loading, exploring, and processing the FAIR^2 dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) Python library.

### Dataset Source
The dataset is defined via a Croissant schema and can be referenced at:
- https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json

In [ ]:
# Install required library
!pip install --quiet mlcroissant

## 1. Data Loading
Load the dataset metadata and records using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Dataset Croissant schema JSON-LD URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load Croissant metadata
dataset = mlc.Dataset(croissant_url)
# The .metadata object contains all top-level dataset-level properties
print(f"Dataset title: {dataset.metadata.name}")
print(f"Description: {dataset.metadata.description}")
print(f"Version: {dataset.metadata.version}")
print(f"License: {dataset.metadata.license}")

## 2. Data Overview
Review available record sets, fields, and columns. All entities are referenced by their `@id` fields to ensure precise references for exploration.

Let's explore the schema structure:

In [ ]:
# List available record sets and their @id
print("Available record sets in the dataset:")
for rs in dataset.metadata.record_sets:
    print(f"- @id: {rs.id} | name: {rs.name}")

In [ ]:
# For this dataset, let's examine all record sets with their fields and columns (all by @id):
record_set_ids = [rs.id for rs in dataset.metadata.record_sets]

for rs in dataset.metadata.record_sets:
    print(f"\nRecord set @id: {rs.id}")
    print(f"  Name: {rs.name}")
    print("  Fields:")
    for field in rs.fields:
        print(f"    - @id: {field.id}, name: {field.name}, data_type: {field.data_type}")
        if hasattr(field, 'columns') and field.columns:
            print("      Columns:")
            for col in field.columns:
                print(f"        - @id: {col.id}, name: {col.name}")

## 3. Data Extraction
We'll load records from the primary record set (the main data table for this dataset) into Pandas DataFrames. All processing is referenced by `@id` fields.

**Note:** For this dataset, there is typically one primary record set for the main protein abundance table. We will extract all, but you may adjust the selected record set as needed.

In [ ]:
# List to store DataFrames for each record set
dataframes = {}

# Extract and load each record set
for rs in dataset.metadata.record_sets:
    print(f"\nLoading records for record set @id: {rs.id} - {rs.name}")
    recs = list(dataset.records(record_set=rs.id))
    df = pd.DataFrame(recs)
    dataframes[rs.id] = df
    print(f"Columns: {list(df.columns)}")
    print("Sample records:")
    display(df.head())

For the following analysis, we'll focus on the main table (record set). Identify the `@id` of the record set containing the protein data. For example, if it is `cr:recordSet1`, use it (you can adjust as needed based on printed output above).

In [ ]:
# Select the main record set by @id
# (Update this to the @id from above if necessary)
main_record_set_id = dataset.metadata.record_sets[0].id  # use the first record set by default
main_df = dataframes[main_record_set_id]
print(f"Working with main record set @id: {main_record_set_id}")

## 4. Exploratory Data Analysis (EDA)
Let's explore the data with typical EDA operations. We select numeric fields by their `@id` (for example, a field like "MW" or "coverage_percentage").

We'll filter, normalize, and group by attributes as an example.

In [ ]:
# Automatically find numeric fields (by data type in schema)
numeric_fields = []
for rs in dataset.metadata.record_sets:
    if rs.id == main_record_set_id:
        for field in rs.fields:
            if field.data_type in ['schema:Float', 'schema:Integer', 'schema:Number', 'Float', 'Integer', 'Number']:
                numeric_fields.append(field.id)
print(f"Numeric fields (@id): {numeric_fields}")
if numeric_fields:
    numeric_field_id = numeric_fields[0]
else:
    raise RuntimeError("No numeric fields found in schema.")
print(f"Will use `{numeric_field_id}` as example numeric field for filtering and normalization.")

# Filter records
threshold = main_df[numeric_field_id].mean()  # Use mean as a threshold
filtered_df = main_df[main_df[numeric_field_id] > threshold]
print(f"Filtered records with {numeric_field_id} > {threshold:.2f}: {len(filtered_df)}")
display(filtered_df.head())

# Normalize selected numeric field
filtered_df[f"{numeric_field_id}_normalized"] = (
    filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
) / filtered_df[numeric_field_id].std()
display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Try to find a categorical/groupable field
group_field_id = None
for rs in dataset.metadata.record_sets:
    if rs.id == main_record_set_id:
        for field in rs.fields:
            if field.data_type in ['schema:Text', 'Text', 'String']:
                # Avoid accession/id fields, prefer a descriptive group
                if 'group' in field.name.lower() or 'category' in field.name.lower() or field.name.lower() in ['sample', 'condition', 'class']:
                    group_field_id = field.id
                    break
# Default: pick second field if no best match
if not group_field_id and len(main_df.columns) > 1:
    group_field_id = main_df.columns[1]

if group_field_id in main_df.columns:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
    print(f"Average {numeric_field_id} grouped by {group_field_id}:")
    display(grouped_df.head())
else:
    print("No suitable grouping field found.")

## 5. Visualization
Let's plot a histogram of the selected numeric field and, if available, a barplot of group means. (Make sure the DataFrames are not empty and numeric data is valid.)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8,4))
sns.histplot(main_df[numeric_field_id].dropna(), bins=30, kde=True)
plt.title(f'Distribution of {numeric_field_id}')
plt.xlabel(numeric_field_id)
plt.ylabel('Frequency')
plt.show()

if group_field_id and group_field_id in main_df.columns:
    plt.figure(figsize=(10,4))
    mean_by_group = main_df.groupby(group_field_id)[numeric_field_id].mean().sort_values(ascending=False).head(20)
    mean_by_group.plot(kind='bar')
    plt.title(f'Mean {numeric_field_id} per {group_field_id} (top 20)')
    plt.ylabel(f'Mean {numeric_field_id}')
    plt.show()

## 6. Conclusion

In this notebook, we've loaded and explored the FAIR^2 dataset using `mlcroissant`, referencing all schema entities by their `@id` for unambiguous processing. We examined available record sets/fields, extracted the main protein abundance records, performed basic filtering and normalization, and visualized the data distribution.

Further analyses may include more advanced statistics, processing additional record sets (if present), and integrating biological knowledge for downstream insights.

For reproducible research, always cite the dataset as:

`Kulka, M., Rodriguez Miera, S., and Marcet-Palacios, M. 2026. Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells. Frontiers.`